# Assignment 1 – Data Quality Analysis

**Course:** SWEN466 Knowledge Discovery and Data Mining  
**Dataset:** Olist – five tables loaded into the `raw` schema via DuckDB

Tables: `customers`, `orders`, `order_items`, `products`, `sellers`

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import pathlib

# ── locate the CSV files (relative to this notebook) ──────────────────────
DATA_DIR = pathlib.Path("../data/raw")

con = duckdb.connect()

# Create the raw schema and load each CSV as a view
con.execute("CREATE SCHEMA IF NOT EXISTS raw")

for table, filename in [
    ("customers",   "customers.csv"),
    ("orders",      "orders.csv"),
    ("order_items", "order_items.csv"),
    ("products",    "products.csv"),
    ("sellers",     "sellers.csv"),
]:
    con.execute(
        f"CREATE OR REPLACE VIEW raw.{table} AS "
        f"SELECT * FROM read_csv_auto('{DATA_DIR / filename}')"
    )

def q(sql: str) -> pd.DataFrame:
    """Run a SQL string and return a pandas DataFrame."""
    return con.execute(sql).df()

print("DuckDB ready – raw schema loaded")

## 1. Explore – DESCRIBE + grain

Run `DESCRIBE` and `SELECT * … LIMIT 5` on every table.
Each table's grain is noted in the comments below.

In [ ]:
# ── customers ──────────────────────────────────────────────────────────────
# GRAIN: one row = one customer–order combination
# (customer_id is unique per order; customer_unique_id identifies the person)
display(q("DESCRIBE raw.customers"))
display(q("SELECT * FROM raw.customers LIMIT 5"))

In [ ]:
# ── orders ─────────────────────────────────────────────────────────────────
# GRAIN: one row = one order placed by a customer
display(q("DESCRIBE raw.orders"))
display(q("SELECT * FROM raw.orders LIMIT 5"))

In [ ]:
# ── order_items ─────────────────────────────────────────────────────────────
# GRAIN: one row = one line-item within an order (order_id + order_item_id)
display(q("DESCRIBE raw.order_items"))
display(q("SELECT * FROM raw.order_items LIMIT 5"))

In [ ]:
# ── products ────────────────────────────────────────────────────────────────
# GRAIN: one row = one product listed in the catalog
display(q("DESCRIBE raw.products"))
display(q("SELECT * FROM raw.products LIMIT 5"))

In [ ]:
# ── sellers ─────────────────────────────────────────────────────────────────
# GRAIN: one row = one registered seller
display(q("DESCRIBE raw.sellers"))
display(q("SELECT * FROM raw.sellers LIMIT 5"))

## 2. Primary keys

For each candidate key: `COUNT(*) = COUNT(DISTINCT key)` and no NULLs.
`order_items` needs a **composite** key `(order_id, order_item_id)`.

In [ ]:
q("""
SELECT
    'customers.customer_id'                  AS candidate_key,
    count(*)                                 AS total_rows,
    count(DISTINCT customer_id)              AS distinct_vals,
    count(*) - count(customer_id)            AS nulls
FROM raw.customers
UNION ALL
SELECT 'orders.order_id',
    count(*), count(DISTINCT order_id),
    count(*) - count(order_id)
FROM raw.orders
UNION ALL
SELECT 'order_items.order_id (alone)',
    count(*), count(DISTINCT order_id),
    count(*) - count(order_id)
FROM raw.order_items
UNION ALL
SELECT 'order_items(order_id, order_item_id)',
    count(*), count(DISTINCT (order_id, order_item_id)),
    count(*) - count(order_id)
FROM raw.order_items
UNION ALL
SELECT 'products.product_id',
    count(*), count(DISTINCT product_id),
    count(*) - count(product_id)
FROM raw.products
UNION ALL
SELECT 'sellers.seller_id',
    count(*), count(DISTINCT seller_id),
    count(*) - count(seller_id)
FROM raw.sellers
""")

**Findings:**
- `customers.customer_id` — unique, no nulls ✓  
- `orders.order_id` — unique, no nulls ✓  
- `order_items.order_id` alone is **not** unique (multiple items per order)  
- `(order_id, order_item_id)` composite — unique, no nulls ✓  
- `products.product_id` — unique, no nulls ✓  
- `sellers.seller_id` — unique, no nulls ✓  

## 3. Cardinality per relationship

For every FK link, compute **min** and **max** child rows per parent row.

In [ ]:
q("""
-- customers  -|---|<  orders  (customer places orders)
SELECT
    'customers → orders'      AS relationship,
    min(child_count)          AS min_children,
    max(child_count)          AS max_children
FROM (
    SELECT c.customer_id, count(o.order_id) AS child_count
    FROM raw.customers c
    LEFT JOIN raw.orders o USING (customer_id)
    GROUP BY c.customer_id
) sub
UNION ALL
-- orders  -|---|<  order_items  (order contains items)
SELECT
    'orders → order_items',
    min(child_count),
    max(child_count)
FROM (
    SELECT o.order_id, count(i.order_id) AS child_count
    FROM raw.orders o
    LEFT JOIN raw.order_items i USING (order_id)
    GROUP BY o.order_id
) sub
UNION ALL
-- products  -|---|<  order_items  (product appears in items)
SELECT
    'products → order_items',
    min(child_count),
    max(child_count)
FROM (
    SELECT p.product_id, count(i.product_id) AS child_count
    FROM raw.products p
    LEFT JOIN raw.order_items i USING (product_id)
    GROUP BY p.product_id
) sub
UNION ALL
-- sellers  -|---|<  order_items  (seller fulfils items)
SELECT
    'sellers → order_items',
    min(child_count),
    max(child_count)
FROM (
    SELECT s.seller_id, count(i.seller_id) AS child_count
    FROM raw.sellers s
    LEFT JOIN raw.order_items i USING (seller_id)
    GROUP BY s.seller_id
) sub
""")

## 4. Foreign key relationships

```
customers 1 ---- N orders 1 ---- N order_items N ---- 1 products
```

A foreign key is healthy when every value exists in the parent table. An **orphan** is
a child row with no parent.

In [ ]:
q("""
SELECT 'orders without customer' AS check_name, count(*) AS n
  FROM raw.orders o ANTI JOIN raw.customers c USING (customer_id)
UNION ALL SELECT 'items without order', count(*)
  FROM raw.order_items i ANTI JOIN raw.orders o USING (order_id)
UNION ALL SELECT 'items without product', count(*)
  FROM raw.order_items i ANTI JOIN raw.products p USING (product_id)
UNION ALL SELECT 'items without seller', count(*)
    FROM raw.order_items i ANTI JOIN raw.sellers s USING (seller_id)
UNION ALL SELECT 'orders without items', count(*)
  FROM raw.orders o ANTI JOIN raw.order_items i USING (order_id)
""")

No orphans in the child-to-parent direction. But 775 orders have no items. An **INNER JOIN** from orders
to items would silently remove them.

## 5. Data Quality

we check five dimensions: completeness(nulls), uniqueness, validity (values in a sensible range),
consistency (fields agree), timeliness (coverage)

### 5.1 Orders with no items

#### Exercise 1

In [ ]:
q("""
SELECT
    o.order_status,
    count(*) AS orders_without_items
FROM raw.orders o
WHERE NOT EXISTS (
    SELECT 1 FROM raw.order_items i WHERE i.order_id = o.order_id
)
GROUP BY o.order_status
ORDER BY orders_without_items DESC
""")

### 5.2 Check order status and class imbalance

#### Exercise 2

In [ ]:
q("""
SELECT
    order_status,
    count(*)                                          AS n,
    round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
FROM raw.orders
GROUP BY order_status
ORDER BY n DESC
""")

### 5.3 Missing and inconsistent timestamps

#### Exercise 3

In [ ]:
q("""
SELECT
    -- completeness: how many NULLs in each timestamp column?
    count(*) FILTER (WHERE order_approved_at IS NULL)              AS approved_null,
    count(*) FILTER (WHERE order_delivered_carrier_date IS NULL)   AS carrier_null,
    count(*) FILTER (WHERE order_delivered_customer_date IS NULL)  AS delivered_null,
    -- consistency: delivered status but no delivery date
    count(*) FILTER (
        WHERE order_status = 'delivered'
          AND order_delivered_customer_date IS NULL
    )                                                              AS delivered_but_no_date,
    -- consistency: carrier handover AFTER customer delivery
    count(*) FILTER (
        WHERE order_delivered_carrier_date > order_delivered_customer_date
    )                                                              AS carrier_after_customer
FROM raw.orders
""")

### 5.4 Time coverage

#### Exercise 4

In [ ]:
monthly = q("""
SELECT
    strftime(order_purchase_timestamp, '%Y-%m') AS month,
    count(*)                                    AS orders
FROM raw.orders
WHERE order_purchase_timestamp IS NOT NULL
GROUP BY month
ORDER BY month
""")
ax = monthly.plot.bar(x="month", y="orders", figsize=(12, 4), legend=False, color="#2a7de1")
ax.set_title("Orders per month: partial months at both ends")
ax.set_xlabel("")
plt.tight_layout(); plt.show()

### 5.5 Product completeness check

#### Exercise 5

In [ ]:
q("""
SELECT
    count(*)                                                    AS products,
    count(*) FILTER (WHERE product_category_name IS NULL)       AS category_null,
    count(*) FILTER (WHERE product_weight_g IS NULL)            AS weight_null,
    count(*) FILTER (WHERE product_weight_g = 0)                AS weight_zero,
    count(DISTINCT product_category_name)                       AS n_categories
FROM raw.products
""")